# 🚗 Travel Sharing Coordination App — ML Notebook
**Fare Prediction Model: EDA, Training & Evaluation**

Project by: Soumyadip Pal, Rohit Paul, Saptarshi Ghosh, Jitendrio Saha
Guide: Prof. Subir Hazra — MSIT, Dept. of IT — Dec 2025

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

plt.style.use('dark_background')
ACCENT = '#00c896'
ACCENT2 = '#0084ff'
ACCENT3 = '#ff6b35'
WARNING = '#f59e0b'
PALETTE = [ACCENT, ACCENT2, ACCENT3, WARNING]

print('✅ Libraries loaded')

## 1. Generate & Load Dataset

In [ ]:
import sys, os
sys.path.append('..')  # add ml/ parent to path

# Generate dataset if not exists
if not os.path.exists('../data/trips_dataset.csv'):
    from generate_dataset import generate_dataset
    df = generate_dataset(5000)
    os.makedirs('../data', exist_ok=True)
    df.to_csv('../data/trips_dataset.csv', index=False)
    print(f'Dataset generated: {len(df):,} records')
else:
    df = pd.read_csv('../data/trips_dataset.csv')
    print(f'Dataset loaded: {len(df):,} records')

df.head()

In [ ]:
print('Shape:', df.shape)
print('\nData Types:')
print(df.dtypes)
print('\nMissing Values:', df.isnull().sum().sum())
print('\nCities:', df['city'].value_counts().to_dict())

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Statistical summary
df[['distance_km','duration_min','traffic_index','surge_flag','actual_fare']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Feature Distributions', fontsize=16, color='white', fontweight='bold')

cols = ['actual_fare','distance_km','duration_min','traffic_index','surge_flag','departure_hour']
colors = [ACCENT, ACCENT2, ACCENT3, WARNING, '#8b5cf6', '#ec4899']

for ax, col, color in zip(axes.flat, cols, colors):
    ax.hist(df[col], bins=40, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(col, color='white')
    ax.set_facecolor('#111827')
    ax.tick_params(colors='gray')
    ax.spines[:].set_color('#1e2d45')
    mean_val = df[col].mean()
    ax.axvline(mean_val, color='white', linestyle='--', alpha=0.6, linewidth=1.2)
    ax.text(mean_val, ax.get_ylim()[1]*0.9, f' μ={mean_val:.1f}', color='white', fontsize=9)

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=120, bbox_inches='tight', facecolor='#0a0f1e')
plt.show()
print('Saved: feature_distributions.png')

In [ ]:
# Fare by city
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fare Analysis by City', fontsize=14, color='white', fontweight='bold')

city_colors = {'Kolkata': ACCENT, 'Delhi': ACCENT2, 'Mumbai': ACCENT3, 'Bengaluru': WARNING}

# Box plot
cities = df['city'].unique()
data_by_city = [df[df['city'] == c]['actual_fare'].values for c in cities]
bp = axes[0].boxplot(data_by_city, labels=cities, patch_artist=True,
                     medianprops={'color':'white','linewidth':2})
for patch, city in zip(bp['boxes'], cities):
    patch.set_facecolor(city_colors[city])
    patch.set_alpha(0.7)
axes[0].set_title('Fare Distribution by City')
axes[0].set_ylabel('Fare (₹)')
axes[0].set_facecolor('#111827')

# Mean fare by hour
for city in cities:
    hourly = df[df['city']==city].groupby('departure_hour')['actual_fare'].mean()
    axes[1].plot(hourly.index, hourly.values, label=city, color=city_colors[city], linewidth=2)
axes[1].set_title('Average Fare by Hour of Day')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Avg Fare (₹)')
axes[1].legend()
axes[1].set_facecolor('#111827')
axes[1].axvspan(7, 9, alpha=0.1, color=ACCENT3, label='Morning peak')
axes[1].axvspan(17, 19, alpha=0.1, color=ACCENT3)

for ax in axes:
    ax.tick_params(colors='gray')
    ax.spines[:].set_color('#1e2d45')

plt.tight_layout()
plt.savefig('fare_by_city.png', dpi=120, bbox_inches='tight', facecolor='#0a0f1e')
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
le_tmp = LabelEncoder()
df['city_enc'] = le_tmp.fit_transform(df['city'])

num_cols = ['distance_km','duration_min','departure_hour','day_of_week',
            'traffic_index','surge_flag','fuel_price','base_fare','city_enc','actual_fare']
corr = df[num_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(10, 133, as_cmap=True)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=cmap,
            linewidths=0.5, linecolor='#1e2d45',
            annot_kws={'size': 9}, ax=ax)

ax.set_title('Feature Correlation Matrix', color='white', fontsize=14, pad=15)
ax.tick_params(colors='gray')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight', facecolor='#0a0f1e')
plt.show()

In [ ]:
# Surge pattern visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Surge & Traffic Patterns', fontsize=14, color='white', fontweight='bold')

# Surge by hour
surge_hourly = df.groupby('departure_hour')['surge_flag'].mean()
axes[0].bar(surge_hourly.index, surge_hourly.values, color=ACCENT3, alpha=0.8)
axes[0].axhline(1.0, color='white', linestyle='--', alpha=0.5, linewidth=1)
axes[0].set_title('Average Surge Multiplier by Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Surge Multiplier')
axes[0].set_facecolor('#111827')

# Traffic index distribution
traffic_counts = df['traffic_index'].value_counts().sort_index()
axes[1].bar([str(x) for x in traffic_counts.index], traffic_counts.values,
            color=[ACCENT2, ACCENT, WARNING, ACCENT3, '#ec4899'], alpha=0.85)
axes[1].set_title('Traffic Index Distribution')
axes[1].set_xlabel('Traffic Index')
axes[1].set_ylabel('Count')
axes[1].set_facecolor('#111827')

for ax in axes:
    ax.tick_params(colors='gray')
    ax.spines[:].set_color('#1e2d45')

plt.tight_layout()
plt.savefig('surge_traffic.png', dpi=120, bbox_inches='tight', facecolor='#0a0f1e')
plt.show()

## 3. Model Training

In [ ]:
# Prepare features
le = LabelEncoder()
df['city_encoded'] = le.fit_transform(df['city'])

features = ['distance_km','duration_min','departure_hour','day_of_week',
            'traffic_index','surge_flag','fuel_price','base_fare','city_encoded']
X = df[features]
y = df['actual_fare']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set    : {X_test.shape[0]:,} samples')
print(f'Features    : {len(features)}')

In [ ]:
# Train all three models
models = {
    'Linear Regression':    (LinearRegression(), True),       # uses scaled data
    'Random Forest':        (RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), False),
    'Gradient Boosting':    (GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=42), False),
}

results = {}
trained_models = {}

for name, (model, use_scaled) in models.items():
    Xtr = X_train_sc if use_scaled else X_train
    Xte = X_test_sc  if use_scaled else X_test
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'preds': preds}
    trained_models[name] = model
    print(f'{name:25s}  MAE=₹{mae:.1f}  RMSE=₹{rmse:.1f}  R²={r2:.3f}')

best_model_name = min(results, key=lambda k: results[k]['MAE'])
print(f'\n✅ Best model: {best_model_name}')

## 4. Model Evaluation & Visualization

In [ ]:
# Side-by-side metric comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, color='white', fontweight='bold')

model_names = list(results.keys())
bar_colors = [ACCENT, ACCENT2, ACCENT3]

for ax, metric, title in zip(axes, ['MAE','RMSE','R2'], ['MAE (₹) ↓','RMSE (₹) ↓','R² Score ↑']):
    vals = [results[m][metric] for m in model_names]
    short = ['LR','RF','GB']
    bars = ax.bar(short, vals, color=bar_colors, alpha=0.85, edgecolor='none', width=0.5)
    ax.set_title(title, color='white')
    ax.set_facecolor('#111827')
    ax.tick_params(colors='gray')
    ax.spines[:].set_color('#1e2d45')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', color='white', fontsize=11, fontweight='bold')
    # highlight best
    best_idx = vals.index(min(vals)) if metric != 'R2' else vals.index(max(vals))
    bars[best_idx].set_edgecolor(ACCENT)
    bars[best_idx].set_linewidth(2.5)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight', facecolor='#0a0f1e')
plt.show()

In [ ]:
# Predicted vs Actual for best model
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Best Model: {best_model_name}', fontsize=14, color='white', fontweight='bold')

preds_best = results[best_model_name]['preds']
sample_idx = np.random.choice(len(y_test), 500, replace=False)
y_sample   = y_test.values[sample_idx]
p_sample   = preds_best[sample_idx]

# Scatter: predicted vs actual
axes[0].scatter(y_sample, p_sample, alpha=0.4, color=ACCENT, s=15)
lims = [min(y_sample.min(), p_sample.min()), max(y_sample.max(), p_sample.max())]
axes[0].plot(lims, lims, 'white', linestyle='--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Fare (₹)', color='gray')
axes[0].set_ylabel('Predicted Fare (₹)', color='gray')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()
axes[0].set_facecolor('#111827')
axes[0].tick_params(colors='gray')
axes[0].spines[:].set_color('#1e2d45')

# Residuals distribution
residuals = y_sample - p_sample
axes[1].hist(residuals, bins=50, color=ACCENT2, alpha=0.8, edgecolor='none')
axes[1].axvline(0, color='white', linestyle='--', linewidth=1.5)
axes[1].axvline(residuals.mean(), color=ACCENT, linestyle='-', linewidth=1.5,
                label=f'Mean residual: ₹{residuals.mean():.1f}')
axes[1].set_xlabel('Residual (Actual − Predicted)', color='gray')
axes[1].set_ylabel('Frequency', color='gray')
axes[1].set_title('Residual Distribution')
axes[1].legend()
axes[1].set_facecolor('#111827')
axes[1].tick_params(colors='gray')
axes[1].spines[:].set_color('#1e2d45')

plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=120, bbox_inches='tight', facecolor='#0a0f1e')
plt.show()

In [ ]:
# Feature Importance (Random Forest & Gradient Boosting)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Feature Importance', fontsize=14, color='white', fontweight='bold')

for ax, name in zip(axes, ['Random Forest', 'Gradient Boosting']):
    importances = trained_models[name].feature_importances_
    sorted_idx  = np.argsort(importances)[::-1]
    sorted_names = [features[i] for i in sorted_idx]
    sorted_vals  = importances[sorted_idx]

    colors_bar = [ACCENT if i == 0 else ACCENT2 if i == 1 else ACCENT3 if i == 2 else '#8b5cf6'
                  for i in range(len(features))]
    ax.barh(sorted_names[::-1], sorted_vals[::-1], color=colors_bar[::-1], alpha=0.85)
    ax.set_title(name)
    ax.set_xlabel('Importance Score', color='gray')
    ax.set_facecolor('#111827')
    ax.tick_params(colors='gray')
    ax.spines[:].set_color('#1e2d45')
    for i, (val, label) in enumerate(zip(sorted_vals[::-1], sorted_names[::-1])):
        ax.text(val + 0.002, i, f'{val:.3f}', va='center', color='white', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight', facecolor='#0a0f1e')
plt.show()

## 5. Cost Sharing & Fare Interval Demo

In [ ]:
# Demo: fare prediction with confidence interval
best = trained_models[best_model_name]
use_scaled_best = best_model_name == 'Linear Regression'

CITY_CONFIG = {
    'Kolkata':   {'base_fare':30,'per_km':13,'per_min':1.2,'fuel_price':103.5},
    'Delhi':     {'base_fare':35,'per_km':15,'per_min':1.5,'fuel_price':94.7},
    'Mumbai':    {'base_fare':40,'per_km':17,'per_min':1.8,'fuel_price':106.3},
    'Bengaluru': {'base_fare':38,'per_km':16,'per_min':1.6,'fuel_price':101.9},
}
SURGE = {0:1.0,1:0.9,2:0.9,3:0.9,4:1.0,5:1.1,6:1.3,7:1.6,8:1.8,9:1.5,
         10:1.2,11:1.1,12:1.3,13:1.2,14:1.1,15:1.2,16:1.4,17:1.7,
         18:1.9,19:1.6,20:1.4,21:1.2,22:1.1,23:1.0}

def predict_fare(city, distance_km, duration_min, hour, dow, traffic=1.0, passengers=2):
    cfg = CITY_CONFIG[city]
    city_enc = le.transform([city])[0]
    surge = SURGE[hour]
    if dow >= 5: surge = max(1.0, surge - 0.2)
    feat = np.array([[distance_km, duration_min, hour, dow, traffic, surge, cfg['fuel_price'], cfg['base_fare'], city_enc]])
    if use_scaled_best: feat = scaler.transform(feat)
    pred = best.predict(feat)[0]
    return {'lower': round(pred*0.85), 'median': round(pred), 'upper': round(pred*1.20), 'per_person': round(pred/passengers)}

# Example predictions
scenarios = [
    ('Kolkata',  7.2, 22, 8,  1, 1.4, 2),
    ('Delhi',    5.8, 18, 18, 2, 1.6, 3),
    ('Mumbai',  12.1, 34, 9,  4, 1.2, 4),
    ('Bengaluru',9.0, 28, 17, 0, 1.5, 2),
]

print(f'{'City':<12} {'Dist':>6} {'Hr':>4} {'Traffic':>8} {'Lower':>8} {'Median':>8} {'Upper':>8} {'Per Person':>12}')
print('-'*80)
for city, dist, dur, hr, dow, trf, pax in scenarios:
    r = predict_fare(city, dist, dur, hr, dow, trf, pax)
    print(f'{city:<12} {dist:>6.1f} {hr:>4} {trf:>8.1f} ₹{r["lower"]:>6} ₹{r["median"]:>6} ₹{r["upper"]:>6} ₹{r["per_person"]:>10} (÷{pax})')

In [ ]:
# Visualise fare intervals across hours for Kolkata
hours = list(range(24))
lowers, medians, uppers = [], [], []

for h in hours:
    r = predict_fare('Kolkata', 8.0, 25, h, 1, 1.2, 2)
    lowers.append(r['lower'])
    medians.append(r['median'])
    uppers.append(r['upper'])

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(hours, lowers, uppers, alpha=0.2, color=ACCENT2, label='Lower–Upper range')
ax.plot(hours, medians, color=ACCENT, linewidth=2.5, label='Median fare', zorder=5)
ax.plot(hours, lowers, color=ACCENT2, linewidth=1, linestyle='--', alpha=0.6)
ax.plot(hours, uppers, color=ACCENT3, linewidth=1, linestyle='--', alpha=0.6)

# Peak zones
ax.axvspan(7, 9,   alpha=0.08, color=ACCENT3, label='Morning peak')
ax.axvspan(17, 19, alpha=0.08, color=ACCENT3, label='Evening peak')

ax.set_title('Kolkata — Predicted Fare Interval by Hour (8 km trip, 2 riders)', color='white', fontsize=13)
ax.set_xlabel('Hour of Day', color='gray')
ax.set_ylabel('Predicted Fare (₹)', color='gray')
ax.set_xticks(hours)
ax.legend()
ax.set_facecolor('#111827')
ax.tick_params(colors='gray')
ax.spines[:].set_color('#1e2d45')
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('₹%d'))

plt.tight_layout()
plt.savefig('fare_by_hour.png', dpi=120, bbox_inches='tight', facecolor='#0a0f1e')
plt.show()

## 6. Cross-Validation & Final Summary

In [ ]:
print('Cross-Validation (5-fold, neg MAE):')
for name, (model, use_scaled) in models.items():
    X_cv = X_train_sc if use_scaled else X_train
    cv_scores = cross_val_score(model, X_cv, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
    print(f'  {name:<25s}  CV-MAE = ₹{-cv_scores.mean():.1f} ± ₹{cv_scores.std():.1f}')

In [ ]:
# Final summary table
summary = pd.DataFrame([
    {'Model': name, 'MAE (₹)': round(r['MAE'],1), 'RMSE (₹)': round(r['RMSE'],1), 'R²': round(r['R2'],3)}
    for name, r in results.items()
])
summary['Deployed'] = summary['Model'].apply(lambda x: '✅ Yes' if x == best_model_name else 'No')
print('\n=== Final Model Summary ===')
print(summary.to_string(index=False))

print(f'\n✅ Best model: {best_model_name}')
print(f'   MAE  = ₹{results[best_model_name]["MAE"]:.1f} (average prediction error)')
print(f'   RMSE = ₹{results[best_model_name]["RMSE"]:.1f}')
print(f'   R²   = {results[best_model_name]["R2"]:.3f}')
print(f'\nModel saved to: ../models/fare_model.pkl')

In [ ]:
# Save all plots summary figure
import pickle, json, os
os.makedirs('../models', exist_ok=True)

with open('../models/fare_model.pkl','wb') as f: pickle.dump(trained_models[best_model_name], f)
with open('../models/scaler.pkl','wb') as f:     pickle.dump(scaler, f)
with open('../models/label_encoder.pkl','wb') as f: pickle.dump(le, f)

meta = {
    'features': features,
    'best_model': best_model_name,
    'cities': list(CITY_CONFIG.keys()),
    'city_config': CITY_CONFIG,
    'surge_table': {str(k): v for k,v in SURGE.items()},
    'metrics': {k: {'MAE': round(v['MAE'],2), 'RMSE': round(v['RMSE'],2), 'R2': round(v['R2'],3)} for k,v in results.items()}
}
with open('../models/meta.json','w') as f: json.dump(meta, f, indent=2)
print('✅ All model artifacts saved to ../models/')